# EfficientNet-B0
Convert audio stems → synthetic mashups → mel spectrograms → CNN classification

The idea is to simulate the noisy mashup test conditions during training by mixing stems from different songs of the same genre and adding ESC-50 noise.

## 1. Setup and Imports

In [ ]:
!pip install librosa timm wandb --quiet

import os
import glob
import random
import warnings
import time
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import librosa
import timm
from sklearn.metrics import f1_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')

# set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {DEVICE}")

## 2. Configuration

In [ ]:
# paths - change these based on your setup
DATA_ROOT = os.path.expanduser("~/data/messy_mashup")  # lightning.ai
# DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"  # kaggle
OUTPUT_DIR = "./outputs"

STEMS_DIR = os.path.join(DATA_ROOT, "genres_stems")
NOISE_DIR = os.path.join(DATA_ROOT, "ESC-50-master", "audio")
TEST_DIR = os.path.join(DATA_ROOT, "mashups")
TEST_CSV = os.path.join(DATA_ROOT, "test.csv")

# audio params
SR = 22050
DURATION = 10.0
TARGET_LEN = int(SR * DURATION)  # 220500 samples
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
FMIN = 20
FMAX = 8000

# genres
GENRES = sorted(['blues', 'classical', 'country', 'disco', 'hiphop',
                 'jazz', 'metal', 'pop', 'reggae', 'rock'])
GENRE2IDX = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE = {i: g for g, i in GENRE2IDX.items()}
STEMS = ['drums', 'vocals', 'bass']  # 'others' is missing for all songs

# training params
SAMPLES_PER_GENRE = 1000  # 10k mashups per epoch
BATCH_SIZE = 32
EPOCHS = 35
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
NUM_WORKERS = 4
MIXUP_ALPHA = 0.4

# stem weights - drums are most informative (from EDA)
STEM_WEIGHTS = {'drums': 0.45, 'vocals': 0.35, 'bass': 0.20}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Will train on {SAMPLES_PER_GENRE * 10} mashups per epoch")

## 3. Weights & Biases Setup

In [ ]:
import wandb

wandb.login(key="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")

run = wandb.init(
    entity="23f3003225-indian-institute-of-technology-madras",
    project="23f3003225-dl-genai-project",
    name="exp_003_cnn_v2",
    config={
        "backbone": "efficientnet_b0",
        "samples_per_genre": SAMPLES_PER_GENRE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "mixup": MIXUP_ALPHA,
    },
    tags=["cnn", "efficientnet"],
)

## 4. Build Data Index

Scan all genre folders and find which stems are available for each song. Then split into train (85%) and val (15%).

In [ ]:
# index all stem files by genre
stem_index = {g: {st: [] for st in STEMS} for g in GENRES}
song_index = {g: [] for g in GENRES}

for genre in GENRES:
    genre_path = os.path.join(STEMS_DIR, genre)
    songs = sorted(s for s in os.listdir(genre_path)
                   if os.path.isdir(os.path.join(genre_path, s)))
    
    for song in songs:
        song_dir = os.path.join(genre_path, song)
        available_stems = []
        for stem in STEMS:
            filepath = os.path.join(song_dir, f"{stem}.wav")
            if os.path.exists(filepath):
                stem_index[genre][stem].append(filepath)
                available_stems.append(stem)
        if available_stems:
            song_index[genre].append({
                'dir': song_dir,
                'stems': available_stems
            })

noise_files = sorted(glob.glob(os.path.join(NOISE_DIR, "*.wav")))
print(f"Found {len(noise_files)} noise clips from ESC-50")

In [ ]:
# 85/15 train-val split
train_stems = {g: {st: [] for st in STEMS} for g in GENRES}
val_songs = {g: [] for g in GENRES}

for genre in GENRES:
    songs = song_index[genre].copy()
    random.shuffle(songs)
    split_point = int(0.85 * len(songs))
    
    train_list = songs[:split_point]
    val_list = songs[split_point:]
    val_songs[genre] = val_list
    
    # only use stems from training songs
    train_dirs = {s['dir'] for s in train_list}
    for stem in STEMS:
        train_stems[genre][stem] = [
            fp for fp in stem_index[genre][stem]
            if os.path.dirname(fp) in train_dirs
        ]
    
    print(f"  {genre}: {len(train_list)} train, {len(val_list)} val")

## 5. Audio Utilities

Using librosa for loading audio and computing mel spectrograms. No torchaudio needed.

In [ ]:
def load_wav(path, sr=SR, target_len=TARGET_LEN):
    # Load a wav file and pad/crop to fixed length
    try:
        y, _ = librosa.load(path, sr=sr, mono=True)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        elif len(y) > target_len:
            start = random.randint(0, len(y) - target_len)
            y = y[start:start + target_len]
        return y.astype(np.float32)
    except:
        return np.zeros(target_len, dtype=np.float32)


def wav_to_mel(y, sr=SR):
    # Convert waveform to log-mel spectrogram
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX
    )
    S_db = librosa.power_to_db(S, ref=np.max, top_db=80)
    return S_db.astype(np.float32)

print("Audio utils ready")

## 6. SpecAugment

Mask random frequency bands and time steps in the spectrogram to make the model more robust.

In [ ]:
def spec_augment(spec, freq_mask=27, time_mask=80, n_freq=2, n_time=2):
    # Apply frequency and time masking to spectrogram
    _, _, n_mels, n_frames = spec.shape
    aug = spec.clone()
    
    # mask frequency bands
    for _ in range(n_freq):
        f = random.randint(0, freq_mask)
        f0 = random.randint(0, max(0, n_mels - f))
        aug[:, :, f0:f0+f, :] = 0
    
    # mask time steps
    for _ in range(n_time):
        t = random.randint(0, time_mask)
        t0 = random.randint(0, max(0, n_frames - t))
        aug[:, :, :, t0:t0+t] = 0
    
    return aug

## 7. Datasets

**MashupDataset**: The key idea — for each sample, pick random stems from different songs 
of the same genre and mix them together. This simulates the noisy mashup test conditions.

We also add ESC-50 noise clips and apply overdrive augmentation.

In [ ]:
class MashupDataset(Dataset):
    # Creates synthetic mashups on-the-fly by mixing stems
    
    def __init__(self, stem_idx, noise_files, samples_per_genre=1000, augment=True):
        self.stem_idx = stem_idx
        self.noise_files = noise_files
        self.augment = augment
        # create sample list (just genre indices)
        self.samples = []
        for genre in GENRES:
            for _ in range(samples_per_genre):
                self.samples.append(GENRE2IDX[genre])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        genre_idx = self.samples[idx]
        genre = IDX2GENRE[genre_idx]

        # pick random stems from DIFFERENT songs of same genre
        stems_wav = []
        for stem_type in STEMS:
            available = self.stem_idx[genre][stem_type]
            if not available:
                continue
            wav = load_wav(random.choice(available))
            # weight stems by importance (drums > vocals > bass)
            gain = random.uniform(0.5, 1.5) * (STEM_WEIGHTS[stem_type] / 0.33)
            stems_wav.append(wav * gain)

        if not stems_wav:
            mel = np.zeros((N_MELS, TARGET_LEN // HOP_LENGTH + 1), dtype=np.float32)
            return torch.from_numpy(mel).unsqueeze(0), genre_idx

        # mix all stems together
        mix = np.sum(stems_wav, axis=0)

        if self.augment:
            # random circular time shift
            mix = np.roll(mix, random.randint(-SR, SR))
            
            # add ESC-50 noise (0-2 clips at random SNR)
            for _ in range(random.randint(0, 2)):
                noise = load_wav(random.choice(self.noise_files))
                snr_db = random.uniform(5.0, 25.0)
                sig_pwr = np.mean(mix ** 2) + 1e-10
                nse_pwr = np.mean(noise ** 2) + 1e-10
                scale = np.sqrt(sig_pwr / (nse_pwr * 10 ** (snr_db / 10)))
                mix = mix + noise * scale
            
            # overdrive effect (30% chance)
            if random.random() < 0.3:
                mix = np.clip(mix * random.uniform(1.2, 3.0), -1, 1)

        # normalize
        peak = np.max(np.abs(mix))
        if peak > 1e-6:
            mix = mix / peak * random.uniform(0.7, 1.0)

        mel = wav_to_mel(mix)
        return torch.from_numpy(mel).unsqueeze(0), genre_idx

In [ ]:
class ValDataset(Dataset):
    # Validation set - mix all stems from each song (no augmentation)
    
    def __init__(self, song_index):
        self.items = []
        for genre in GENRES:
            for song in song_index[genre]:
                self.items.append((song, GENRE2IDX[genre]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        song_info, label = self.items[idx]
        stems = [load_wav(os.path.join(song_info['dir'], f"{st}.wav"))
                 for st in song_info['stems']]
        mix = np.sum(stems, axis=0)
        peak = np.max(np.abs(mix))
        if peak > 1e-6:
            mix = mix / peak
        mel = wav_to_mel(mix)
        return torch.from_numpy(mel).unsqueeze(0), label


class TestDataset(Dataset):
    # Test mashups for generating predictions
    
    def __init__(self, test_dir, test_csv):
        self.df = pd.read_csv(test_csv, dtype={'id': str})
        self.paths = []
        for _, row in self.df.iterrows():
            path = None
            for pattern in [f"song{str(row['id']).zfill(4)}.wav",
                          f"{row['id']}.wav",
                          f"song{row['id']}.wav"]:
                p = os.path.join(test_dir, pattern)
                if os.path.exists(p):
                    path = p
                    break
            self.paths.append(path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.paths[idx]
        if path:
            mel = wav_to_mel(load_wav(path))
            mel_tensor = torch.from_numpy(mel).unsqueeze(0)
        else:
            mel_tensor = torch.zeros(1, N_MELS, TARGET_LEN // HOP_LENGTH + 1)
        return mel_tensor, str(self.df.iloc[idx]['id'])

print("Datasets ready")

## 8. Model Architecture

**EfficientNet-B0** pretrained on ImageNet, adapted for single-channel mel spectrograms.

Key components:
- **Instance Normalization**: makes model volume-invariant (classical is 20x quieter than hiphop)
- **GeM Pooling**: focuses on high-activation regions instead of averaging everything
- **SpecAugment**: applied during training for regularization

In [ ]:
class GeM(nn.Module):
    # Generalized Mean Pooling - focuses on discriminative regions
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p))
        self.eps = eps
    
    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=(-2, -1)).pow(1.0 / self.p)


class GenreClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.inst_norm = nn.InstanceNorm2d(1)  # volume normalization
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=True,
            in_chans=1, num_classes=0, global_pool=''
        )
        num_features = self.backbone.num_features  # 1280 for effnet-b0
        self.gem = GeM(p=3.0)
        self.head = nn.Sequential(
            nn.LayerNorm(num_features),
            nn.Dropout(0.5),
            nn.Linear(num_features, num_classes)
        )

    def forward(self, x, augment=False):
        x = self.inst_norm(x)
        if augment:
            x = spec_augment(x)
        features = self.backbone(x)
        pooled = self.gem(features)
        return self.head(pooled)


# quick sanity check
model = GenreClassifier().to(DEVICE)
dummy = torch.randn(2, 1, N_MELS, TARGET_LEN // HOP_LENGTH + 1).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"Output shape: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
del dummy, out
gc.collect()
torch.cuda.empty_cache()

## 9. Mixup Augmentation

Mixup blends two training samples and their labels. Helps with generalization.

In [ ]:
def mixup_data(x, y, alpha=0.4):
    # Mix two samples with random weight from Beta distribution
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    # Weighted loss for mixed labels
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

## 10. Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    total_loss = 0
    num_samples = 0
    
    for mel, labels in tqdm(loader, desc="Train", leave=False):
        mel = mel.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()

        # apply mixup 50% of the time
        if random.random() < 0.5:
            mel, y_a, y_b, lam = mixup_data(mel, labels, MIXUP_ALPHA)
            with autocast():
                logits = model(mel, augment=True)
                loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
        else:
            with autocast():
                logits = model(mel, augment=True)
                loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item() * len(labels)
        num_samples += len(labels)
    
    return total_loss / num_samples


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    for mel, labels in loader:
        mel = mel.to(DEVICE)
        with autocast():
            logits = model(mel, augment=False)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return f1, acc, np.array(all_preds), np.array(all_labels)

## 11. Training Loop

In [ ]:
val_ds = ValDataset(val_songs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
print(f"Validation samples: {len(val_ds)}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = GradScaler()

best_f1 = 0.0
history = {'loss': [], 'val_f1': [], 'val_acc': [], 'lr': []}

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    # regenerate mashups every epoch (different combinations each time!)
    train_ds = MashupDataset(train_stems, noise_files, SAMPLES_PER_GENRE, augment=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

    loss = train_one_epoch(model, train_loader, optimizer, scaler, criterion)
    scheduler.step()
    
    val_f1, val_acc, _, _ = evaluate(model, val_loader)
    lr = scheduler.get_last_lr()[0]
    elapsed = time.time() - start_time

    # track history
    history['loss'].append(loss)
    history['val_f1'].append(val_f1)
    history['val_acc'].append(val_acc)
    history['lr'].append(lr)
    wandb.log({"epoch": epoch, "loss": loss, "val_f1": val_f1,
               "val_acc": val_acc, "lr": lr})

    # save best model
    tag = ""
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_cnn.pth'))
        tag = " ★"

    print(f"E{epoch:02d}/{EPOCHS} | loss={loss:.4f} | f1={val_f1:.4f} | "
          f"acc={val_acc:.4f} | lr={lr:.6f} | {elapsed:.0f}s{tag}")

print(f"\nBest validation F1: {best_f1:.4f}")

## 12. Results and Plots

In [ ]:
# training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['loss'])
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_f1'], label='F1')
axes[1].plot(history['val_acc'], label='Accuracy', alpha=0.7)
axes[1].set_title('Validation Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(history['lr'])
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Epoch')

plt.suptitle(f'EfficientNet-B0 — Best F1: {best_f1:.4f}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
wandb.log({"plots/curves": wandb.Image(fig)})
plt.show()

In [ ]:
# confusion matrix on best model
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_cnn.pth'), weights_only=True))
val_f1, val_acc, preds, labels = evaluate(model, val_loader)

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(labels, preds)
ConfusionMatrixDisplay(cm, display_labels=GENRES).plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title(f'Confusion Matrix — F1={val_f1:.4f}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
wandb.log({"plots/confusion": wandb.Image(fig)})
plt.show()

print(classification_report(labels, preds, target_names=GENRES))

## 13. Inference on Test Set

Using 5x TTA (test-time augmentation) — run inference 5 times and average the probabilities for more robust predictions.

In [ ]:
@torch.no_grad()
def predict_tta(model, loader, n_tta=5):
    # Run inference multiple times and average predictions
    model.eval()
    
    # get all IDs first
    all_ids = []
    for _, ids in loader:
        all_ids.extend(ids)

    # run n_tta forward passes
    all_probs = []
    for tta_round in range(n_tta):
        round_probs = []
        for mel, _ in loader:
            mel = mel.to(DEVICE)
            with autocast():
                logits = model(mel, augment=False)
            round_probs.append(F.softmax(logits, dim=1).cpu().numpy())
        all_probs.append(np.vstack(round_probs))

    # average across TTA rounds
    avg_probs = np.mean(all_probs, axis=0)
    return avg_probs.argmax(1), all_ids, avg_probs


# run inference
test_ds = TestDataset(TEST_DIR, TEST_CSV)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
print(f"Test samples: {len(test_ds)}")

preds, ids, probs = predict_tta(model, test_loader, n_tta=5)
print(f"Prediction distribution: {Counter(preds)}")

## 14. Generate Submission

In [ ]:
# create submission csv
test_df = pd.read_csv(TEST_CSV, dtype={'id': str})
pred_dict = {str(id_): IDX2GENRE[p] for id_, p in zip(ids, preds)}
test_df['genre'] = test_df['id'].apply(lambda x: pred_dict.get(str(x), 'rock'))
test_df[['id', 'genre']].to_csv(os.path.join(OUTPUT_DIR, 'submission.csv'), index=False)

print("Submission saved!")
print(test_df['genre'].value_counts().sort_index())

# save probabilities for later ensembling
np.save(os.path.join(OUTPUT_DIR, 'test_probs_efficientnet.npy'), probs)
print("\nProbabilities saved for ensemble use")

## 15. Finish

In [ ]:
wandb.log({"best_f1": best_f1, "status": "complete"})

# save model as artifact
artifact = wandb.Artifact("cnn_efficientnet_v2", type="model")
artifact.add_file(os.path.join(OUTPUT_DIR, 'best_cnn.pth'))
run.log_artifact(artifact)

wandb.finish()
print(f"Done! Best val F1 = {best_f1:.4f}")